In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import json
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')


In [4]:
# (CHANGE THESE ACCORDING TO YOUR DRIVE)
STUDENTLIFE_PATH = "/content/drive/MyDrive/AI Project - Academic Burnout/Data"
#OUTPUT_PATH = "/content/drive/MyDrive/AI_Burnout_Predictor/results_realistic_studentlife"

#os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Paths configured.")


Paths configured.


In [5]:
# Loading StudentLife data
def load_studentlife_json(folder_path):
    all_data = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".json"):
            student_id = file_name.replace(".json", "")
            with open(os.path.join(folder_path, file_name), "r") as f:
                records = json.load(f)
                for r in records:
                    r["student_id"] = student_id
                    all_data.append(r)
    return pd.DataFrame(all_data)

print("Loading StudentLife...")

stress_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Stress"))
activity_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Activity"))
sleep_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Sleep"))

print(f"Stress: {stress_raw.shape}")
print(f"Activity: {activity_raw.shape}")
print(f"Sleep: {sleep_raw.shape}")


Loading StudentLife...
Stress: (2408, 5)
Activity: (833, 9)
Sleep: (1644, 7)


In [ ]:
stress_raw.head()

,null,resp_time,student_id,level,location
0,3,1364121467,Stress_u20,NaN,NaN
1,"43.70413179,-72.28882107",1364121469,Stress_u20,NaN,NaN
2,1,1364121470,Stress_u20,NaN,NaN
3,"43.70413179,-72.28882107",1364121793,Stress_u20,NaN,NaN
4,2,1364121465,Stress_u20,NaN,NaN


In [ ]:
activity_raw.head()

,Social2,null,resp_time,student_id,other_relaxing,other_working,relaxing,working,location
0,2,4,1364884639,Activity_u33,NaN,NaN,NaN,NaN,NaN
1,3,2,1364590835,Activity_u33,NaN,NaN,NaN,NaN,NaN
2,2,1,1364504673,Activity_u33,NaN,NaN,NaN,NaN,NaN
3,2,2,1364677565,Activity_u33,NaN,NaN,NaN,NaN,NaN
4,2,2,1364765272,Activity_u33,NaN,NaN,NaN,NaN,NaN


In [ ]:
sleep_raw.head()

,hour,location,rate,resp_time,social,student_id,null
0,9,"43.70357146,-72.29017646",1,1364761981,1,Sleep_u22,NaN
1,NaN,NaN,NaN,1364122237,NaN,Sleep_u22,6
2,NaN,NaN,NaN,1364122241,NaN,Sleep_u22,"43.70629505,-72.28825598"
3,NaN,NaN,NaN,1364122243,NaN,Sleep_u22,9
4,NaN,NaN,NaN,1364122235,NaN,Sleep_u22,6


In [6]:
# Cleaning the STRESS DATASET
# =========================
print("\n DATASET: STRESS")
print("Shows self-reported student stress levels over time")

print("\nCleaning StudentLife Stress...")

stress_clean = stress_raw.copy()

# Dropping the 'null' column
if 'null' in stress_clean.columns:
    stress_clean = stress_clean.drop(columns=['null'])

print("\nInitial Stress Dataset:")
print(stress_clean)

# Converting  timestamp
stress_clean['timestamp'] = pd.to_datetime(stress_clean['resp_time'], unit='s')
print("\nAfter converting resp_time to timestamp:")
print(stress_clean)

# Cleaning  student_id
stress_clean['student_id'] = stress_clean['student_id'].str.replace('Stress_', '')
print("\nAfter cleaning student_id:")
print(stress_clean)



 DATASET: STRESS
Shows self-reported student stress levels over time

Cleaning StudentLife Stress...

Initial Stress Dataset:
     level                              location   resp_time  student_id
0        3  43.70712148268646,-72.29087061546424  1364681195  Stress_u49
1      NaN                                   NaN  1364121911  Stress_u49
2      NaN                                   NaN  1364121426  Stress_u49
3      NaN                                   NaN  1364121429  Stress_u49
4      NaN                                   NaN  1364118814  Stress_u49
...    ...                                   ...         ...         ...
2403     2              43.70390811,-72.29065789  1369368560  Stress_u16
2404     2              43.70675104,-72.28740962  1369349334  Stress_u16
2405     1              43.70372826,-72.29097234  1369428881  Stress_u16
2406     3              43.68753666,-72.29250241  1369719704  Stress_u16
2407     3              43.68753666,-72.29250241  1369729103  Stress_u

In [8]:
#renaming the level column
stress_clean = stress_clean.rename(columns={'level': 'stress_level'})
print("\nAfter renaming level → stress_level:")
print(stress_clean)

#Converting stress_level to numeric data
stress_clean['stress_level'] = pd.to_numeric(stress_clean['stress_level'], errors='coerce')
print("\nAfter converting stress_level to numeric:")
print(stress_clean)

if 'null' in stress_clean.columns:

    # if stress_level is missing but null looks like numeric, then use this
    null_as_num = pd.to_numeric(stress_clean['null'], errors='coerce')
    fill_mask = stress_clean['stress_level'].isna() & null_as_num.notna()
    if fill_mask.any():
        stress_clean.loc[fill_mask, 'stress_level'] = null_as_num.loc[fill_mask]

    #If location is missing but null looks like "lat,long", then use this
    if 'location' in stress_clean.columns:
        null_as_str = stress_clean['null'].astype(str)
        coord_mask = stress_clean['location'].isna() & null_as_str.str.match(
            r'^-?\d+(\.\d+)?,-?\d+(\.\d+)?$'
        )
        if coord_mask.any():
            stress_clean.loc[coord_mask, 'location'] = stress_clean.loc[coord_mask, 'null']

print("\nAfter recovering values from 'null' (if applicable):")
print(stress_clean)

#dropping missing stress values
stress_clean = stress_clean.dropna(subset=['stress_level'])
print("\nAfter dropping NaN stress levels:")
print(stress_clean)

#Explicit float conversion
stress_clean['stress_level'] = stress_clean['stress_level'].astype(float)
print("\nFinal cleaned Stress dataset:")
print(stress_clean)



After renaming level → stress_level:
      stress_level                              location   resp_time  \
0              3.0  43.70712148268646,-72.29087061546424  1364681195   
6              3.0  43.70286921804329,-72.29096447310383  1364594494   
7              3.0  43.70697950348372,-72.29051295213978  1364536874   
8              3.0  43.70711283219849,-72.29060053949401  1364712773   
9              4.0  43.70700745335506,-72.29084036869996  1364798130   
...            ...                                   ...         ...   
2403           2.0              43.70390811,-72.29065789  1369368560   
2404           2.0              43.70675104,-72.28740962  1369349334   
2405           1.0              43.70372826,-72.29097234  1369428881   
2406           3.0              43.68753666,-72.29250241  1369719704   
2407           3.0              43.68753666,-72.29250241  1369729103   

     student_id           timestamp  
0           u49 2013-03-30 22:06:35  
6           u49 2013-